#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
LEVEL4_BUCKET_PATH = PARENT / "server/out/places_level4"
LEVEL5_BUCKET_PATH = PARENT / "server/out/places_level5"
LEVEL5_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL4_BUCKET = [f for f in LEVEL4_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL4 = pd.concat([pd.read_csv(f) for f in LEVEL4_BUCKET], ignore_index=True)

#### Build Region Grid

In [2]:
from server.scripts.rank_places_parse_locality.get_local import get_local_tile_id
from server.scripts.h3.h3_api import _h3_get_neighbours

df_level5 = DF_LEVEL4.copy()
df_level5['local_tile'] = df_level5.apply(lambda row: get_local_tile_id(row), axis=1)

grid_res9 = pd.DataFrame({
    'local_tile': df_level5['local_tile'].unique(),
}).sort_values('local_tile').reset_index(drop=True)

grid_res9['local_tiles'] = grid_res9['local_tile'].apply(
    lambda tile: _h3_get_neighbours(tile, k=2) | {tile}  # include self
)

if not (MAP_PATH / "grid_res9.csv").exists():
    for row in grid_res9.itertuples():
        places = df_level5[df_level5['local_tile'] == row.local_tile]
        grid_res9.at[row.Index, 'density'] = len(places)
        local_places = df_level5[df_level5['local_tile'].isin(row.local_tiles)]
        grid_res9.at[row.Index, 'local_density'] = len(local_places)

    grid_res9.to_csv(MAP_PATH / "grid_res9.csv", index=False)

#### Global Score

In [3]:
from server.scripts.rank_places_parse_locality.get_local import TYPE_COL

# ── Global type distribution P(T | global) ────────────────────────────────────
global_composition = (
    df_level5[TYPE_COL].value_counts(normalize=True)
    .rename("p_global")
    .reset_index()
    .rename(columns={"index": TYPE_COL})
)

#### Locality Score

In [4]:
from server.scripts.rank_places_parse_locality.get_local import get_local_competition_factor
rep_ratio_per_cell = []
for _, region_row in grid_res9.iterrows():
    local_composition = get_local_competition_factor(
        df_level5, 
        global_composition, 
        region_row['local_tiles'], 
        region_row['local_tile'],
    )
    if local_composition is not None:
        rep_ratio_per_cell.append(local_composition)
# rr - Representation Ratio
grid_df = pd.concat(rep_ratio_per_cell, ignore_index=True)
grid_df.to_csv(MAP_PATH / "grid_res9_compositions.csv", index=False)

In [5]:
# ── Join credible competition_factor back onto df_level5 ──────────────────────
df_level5_2 = df_level5.merge(
    grid_df[["local_tile", TYPE_COL, "p_local", "p_credible", "competition_factor",
           "representations", "neighbour_count"]],
    on=["local_tile", TYPE_COL],
    how="left"
).sort_values(["local_tile"]).reset_index(drop=True)
df_level5_2["competition_factor"] = df_level5_2["competition_factor"].fillna(1.0)
df_level5_2["p_credible"]         = df_level5_2["p_credible"].fillna(0.0)
df_level5_2["representations"]    = df_level5_2["representations"].fillna(0).astype(int)
df_level5_2["neighbour_count"]    = df_level5_2["neighbour_count"].fillna(0).astype(int)
display(df_level5_2.head(2))

,id,displayName,primaryTypeDisplayName,pcd,lat,lon,rating,userRatingCount,shortFormattedAddress,areacode,...,medianPrice,capped_ratings,wilson_score,wilson_quantile,local_tile,p_local,p_credible,competition_factor,representations,neighbour_count
0,ChIJnVjG4_MGdkgRPv420tn5MV8,Taste of China,Takeout Restaurant,SW16 4TR,51.404948,-0.133488,3.4,117.0,"241 Northborough Rd, London",SW16,...,15.0,117.0,0.509415,0.133279,89194ac3497ffff,0.250000,0.030062,1.0,1,4
1,ChIJD8x54k4DdkgRh_HUL2HZjfY,Cooking Happy,Thai Restaurant,SE15 3EJ,51.454419,-0.049200,5.0,28.0,"163 Athenlay Rd, London",SE15,...,35.0,28.0,0.879353,0.777476,89194ad000fffff,0.142857,0.016999,1.0,1,7


#### Adjust Score

In [6]:
import numpy as np
from server.scripts.rank_places_parse_locality.get_local import Z_CONFIDENCE, ALPHA

# competition_boost: only amplify over-represented (competition_factor > 1), neutral otherwise.
# clip(lower=1) → no penalty for rare cuisines; they may be hidden gems.
df_level5_2["competition_boost"] = 1 + ALPHA * np.log(df_level5_2["competition_factor"].clip(lower=1))
df_level5_2["adjusted_score"]    = df_level5_2["wilson_score"] * df_level5_2["competition_boost"]
df_level5_2["adjusted_quantile"] = df_level5_2["adjusted_score"].rank(pct=True)
df_level5_2 = df_level5_2.sort_values("adjusted_quantile", ascending=False).reset_index(drop=True)

print(f"Places: {len(df_level5_2)}  |  Regions: {grid_res9.shape[0]}  |  α={ALPHA}  |  z={Z_CONFIDENCE}")
df_level5_2.head(5)[[
    "displayName", TYPE_COL, "representations", 
    "neighbour_count", "rating", "userRatingCount",
    "p_local", "p_credible", "competition_boost", 
    "wilson_score", "adjusted_score", "adjusted_quantile"
]]


Places: 12320  |  Regions: 1516  |  α=0.2  |  z=2.576


,displayName,cuisineType,representations,neighbour_count,rating,userRatingCount,p_local,p_credible,competition_boost,wilson_score,adjusted_score,adjusted_quantile
0,Ormer Mayfair,Fine Dining,23,279,4.8,521.0,0.082437,0.049110,1.387877,0.927771,1.287633,1.000000
1,Pavyllon London,Fine Dining,15,135,4.7,639.0,0.111111,0.058909,1.424263,0.901941,1.284601,0.999919
2,Labombe by Trivet,Fine Dining,15,135,4.9,60.0,0.111111,0.058909,1.424263,0.898629,1.279884,0.999838
3,The Ritz Restaurant,Fine Dining,23,279,4.7,1385.0,0.082437,0.049110,1.387877,0.906999,1.258803,0.999756
4,Bella Cafe,African,15,130,5.0,65.0,0.115385,0.061221,1.329797,0.944197,1.255589,0.999675


#### Export

In [7]:
for seed_id, group in df_level5_2.groupby("seed_index"):
    save_path = LEVEL5_BUCKET_PATH / f"{seed_id}.csv"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)

df_level5_2.to_csv(LEVEL5_BUCKET_PATH.parent / "places.csv", index=False)

## SAMPLE

In [31]:
SAMPLE = df_level5_2.copy()
SAMPLE.sort_values("wilson_score", ascending=False, inplace=True)
SAMPLE = SAMPLE[[
    "displayName", TYPE_COL, "representations", 
    "rating", "userRatingCount",
    "p_local", "wilson_score", "adjusted_score", 
    "adjusted_quantile", "areacode"
]]
# SAMPLE = SAMPLE[SAMPLE['areacode'] == 'SW12']
# SAMPLE[SAMPLE['userRatingCount']>= 1000]
SAMPLE.reset_index(drop=True, inplace=True)
SAMPLE[SAMPLE['displayName'].str.contains("Milk", case=False, na=False)]

,displayName,cuisineType,representations,rating,userRatingCount,p_local,wilson_score,adjusted_score,adjusted_quantile,areacode
1605,Milk Beach Soho,Australian,2,4.7,1548.0,0.001862,0.906999,0.906999,0.848498,W1D
1658,Tigermilk - Tottenham Court Road,Unspecified,123,4.7,4492.0,0.119534,0.906999,0.906999,0.848498,WC2H
1968,Tigermilk | Spitalfields,Unspecified,82,4.7,499.0,0.135987,0.898503,0.898503,0.814610,E1
3931,"DZRT Canary Wharf | Desserts, Milkshakes, Waff...",Dessert & Ice Cream,2,4.6,211.0,0.011696,0.852099,0.852099,0.658604,E14
4063,Milk Beach,Australian,1,4.5,729.0,0.030303,0.849009,0.849009,0.647930,NW6
11897,"Milky""s shakes & dessert",Dessert & Ice Cream,1,5.0,1.0,0.034483,0.206543,0.206543,0.037094,SW17
